In [1]:
import datetime
import gzip
import json
from tqdm import tqdm
import random
import os
import csv

dataset_name = "Books" # CDs_and_Vinyl, Toys_and_Games, Video_Games, Musical_Instruments, Books
meta_path = f"{dataset_name}/raw/meta_{dataset_name}.json.gz"
inter_path = f"{dataset_name}/raw/{dataset_name}_5.json.gz"
k_core = 5
    

def get_timestamp_start(year, month):
    return int(datetime.datetime(year=year, month=month, day=1, hour=0, minute=0, second=0, microsecond=0).timestamp())

def process(category, metadata=None, reviews=None, st_year=2017, st_month=10, ed_year=2018, ed_month=11, output=True):
    if st_year < 1996:
        return
    start_timestamp = get_timestamp_start(st_year, st_month)
    end_timestamp = get_timestamp_start(ed_year, ed_month)
    if metadata is None:
        with gzip.open(meta_path, 'r') as f:
            metadata = [json.loads(line) for line in f]
        with gzip.open(inter_path, 'r') as f:
            reviews = [json.loads(line) for line in f]
    else:
        metadata = metadata
        reviews = reviews
    
    # item: asin, title, price, imUrl, related, salesRank, categories, description
    users = set()
    items = set()
    for review in tqdm(reviews):
        if int(review["unixReviewTime"]) < start_timestamp or int(review["unixReviewTime"]) > end_timestamp:
            continue        
        users.add(review['reviewerID'])
        items.add(review['asin'])

    remove_users = set()
    remove_items = set()
    
    
    id_title = {}
    for meta in tqdm(metadata):
        if ('title' not in meta) or (meta['title'].find('<span id') > -1):
            remove_items.add(meta['asin'])
            continue
        meta['title'] = meta["title"].replace("&quot;", "\"").replace("&amp;", "&").strip(" ").strip("\"")
        if len(meta['title']) > 1 and len(meta['title'].split(" ")) <= 20: # remove the item without title # remove too long title
            id_title[meta['asin']] = meta['title']
        else:
            remove_items.add(meta['asin'])
    for review in tqdm(reviews):
        if review['asin'] not in id_title:
            remove_items.add(review['asin'])

    while True:
        new_reviews = []
        flag = False
        total = 0
        users = dict()
        items = dict()
        new_reviews = []
        for review in tqdm(reviews):
            if int(review["unixReviewTime"]) < start_timestamp or int(review["unixReviewTime"]) > end_timestamp:
                continue
            if review['reviewerID'] in remove_users or review['asin'] in remove_items:
                continue          
            if review['reviewerID'] not in users:
                users[review['reviewerID']] = 0
            users[review['reviewerID']] += 1
            if review['asin'] not in items:
                items[review['asin']] = 0
                
            items[review['asin']] += 1
            total += 1
            new_reviews.append(review)
            
        for user in users:
            if users[user] < k_core:
                remove_users.add(user)
                flag = True
        
        for item in items:
            if items[item] < k_core:
                remove_items.add(item)
                flag = True
        if st_year > 1996 and len(items) < 5000:
            break
        if not flag:
            break

    if st_year > 1996 and len(items) < 5000:
        if st_month - 3 <= 0:
            st_month = st_month + 12 - 3
            st_year -= 1
        else:
            st_month -= 3
        process(category, metadata = metadata, reviews = reviews, st_year=st_year, st_month=st_month, ed_year=ed_year, ed_month=ed_month, output=True)
        return
        

    reviews = new_reviews
    # shuffle items and assign the id to each item
    items = list(items.keys())
    random.seed(42)
    random.shuffle(items)
    item2id = dict()
    count = 0
    if not os.path.exists(f"{dataset_name}/info"):
        os.mkdir(f"{dataset_name}/info")
    with open(f"{dataset_name}/info/{category}_{k_core}_{st_year}-{st_month}-{ed_year}-{ed_month}.txt", 'w') as f:
        for item in items:
            item2id[item] = count
            f.write(f"{id_title[item]}\t{count}\n")
            count += 1
            
    if not output:
        return 
    # get all the review data

    interact = dict()
    old_items = set(items)
    new_items = set()
    all_inter_num = 0
    for review in tqdm(new_reviews):
        user = review['reviewerID']
        item = review['asin']
        if user not in interact:
            interact[user] = {
                'items': [],
                'ratings': [],
                'timestamps': [],
                'reviews': []
            }
        new_items.add(item)
        interact[user]['items'].append(item)
        interact[user]['ratings'].append(review['overall'])
        interact[user]['timestamps'].append(review['unixReviewTime'])
        all_inter_num += 1
    interaction_list = []   
    for key in tqdm(interact.keys()):
        items = interact[key]['items']
        ratings = interact[key]['ratings']
        timestamps = interact[key]['timestamps']
        all = list(zip(items, ratings, timestamps))
        res = sorted(all, key=lambda x: int(x[2]))
        items, ratings, timestamps = zip(*res)
        items, ratings, timestamps = list(items), list(ratings), list(timestamps)
        interact[key]['items'] = items
        interact[key]['ratings'] = ratings
        interact[key]['timestamps'] = timestamps
        interact[key]['item_ids'] = [item2id[item] for item in items]
        interact[key]['title'] = [id_title[item] for item in items]
        for i in range(1, len(items)):
            st = max(i - 10, 0)
            interaction_list.append([key, interact[key]['items'][st:i], interact[key]['items'][i], interact[key]['item_ids'][st:i], interact[key]['item_ids'][i], interact[key]['title'][st:i], interact[key]['title'][i], interact[key]['ratings'][st:i], interact[key]['ratings'][i], interact[key]['timestamps'][st:i], interact[key]['timestamps'][i]])
    

    # split train valid test
    interaction_list = sorted(interaction_list, key=lambda x: int(x[-1]))
    if not os.path.exists(f"{dataset_name}/train"):
        os.mkdir(f"{dataset_name}/train")
    if not os.path.exists(f"{dataset_name}/valid"):
        os.mkdir(f"{dataset_name}/valid")
    if not os.path.exists(f"{dataset_name}/test"):
        os.mkdir(f"{dataset_name}/test")
    with open(f"{dataset_name}/train/{category}_{k_core}_{st_year}-{st_month}-{ed_year}-{ed_month}.csv", "w") as f:
        writer = csv.writer(f)
        writer.writerow(['user_id', 'item_asins', 'item_asin', 'history_item_id', 'item_id', 'history_item_title', 'item_title', 'history_rating', 'rating', 'history_timestamp', 'timestamp'])
        writer.writerows(interaction_list[:int(len(interaction_list) * 0.8)])
    with open(f"{dataset_name}/valid/{category}_{k_core}_{st_year}-{st_month}-{ed_year}-{ed_month}.csv", "w") as f:
        writer = csv.writer(f)
        writer.writerow(['user_id', 'item_asins', 'item_asin', 'history_item_id', 'item_id', 'history_item_title', 'item_title', 'history_rating', 'rating', 'history_timestamp', 'timestamp'])
        writer.writerows(interaction_list[int(len(interaction_list) * 0.8):int(len(interaction_list) * 0.9)])
    with open(f"{dataset_name}/test/{category}_{k_core}_{st_year}-{st_month}-{ed_year}-{ed_month}.csv", "w") as f:
        writer = csv.writer(f)
        writer.writerow(['user_id', 'item_asins', 'item_asin', 'history_item_id', 'item_id', 'history_item_title', 'item_title', 'history_rating', 'rating', 'history_timestamp', 'timestamp'])
        writer.writerows(interaction_list[int(len(interaction_list) * 0.9):])

    print(f"user_num: {len(interact)}, item_num: {len(new_items)}, inter_num: {all_inter_num}, sample_num: {len(interaction_list)}")
    print(f"Train {category}: {len(interaction_list[:int(len(interaction_list) * 0.8)])}")
    print(f"Valid {category}: {len(interaction_list[int(len(interaction_list) * 0.8):int(len(interaction_list) * 0.9)])}")
    print(f"Test {category}: {len(interaction_list[int(len(interaction_list) * 0.9):])}")
    print("Done!")
process(dataset_name, st_year=2017, st_month=10, ed_year=2018, ed_month=11)

100%|██████████| 67708/67708 [00:15<00:00, 4334.41it/s] 


user_num: 67708, item_num: 41722, inter_num: 921455, sample_num: 853747
Train Books: 682997
Valid Books: 85375
Test Books: 85375
Done!


In [3]:
import glob
import os
import pandas as pd

dataset_name = "CDs_and_Vinyl" # CDs_and_Vinyl, Toys_and_Games, Video_Games, Musical_Instruments, Books
dataset_type = "valid"  # train, valid, test

csv_files = glob.glob(f"{dataset_name}/{dataset_type}/{dataset_name}*.csv")
assert len(csv_files) == 1, "There should be only one csv file in the directory."
csv_file = csv_files[0]

df = pd.read_csv(csv_file)
df.to_json(f'{dataset_name}/{dataset_type}/extra.json', orient='records', indent=4)
